# Code Interpreter

*Notebook 10*

Let your agent write and run Python code in an isolated hosted sandbox.

Executed code helps, but incorrect code can still produce confident errors.

---

## 🔧 Setup

This notebook starts five paid agent runs.

Four use Code Interpreter and start four separate auto-container sessions.

In [ ]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Image, display

load_dotenv(dotenv_path=Path("..") / ".env")

from openai import OpenAI
from agents import (
    Agent,
    CodeInterpreterTool,
    MessageOutputItem,
    Runner,
    ToolCallItem,
)

MODEL = "gpt-5-mini"

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
created_file_ids = []
created_container_ids = []

print("✅ Ready!")

#### Helpers: Execution Evidence and File Retrieval

In [ ]:
def get_code_interpreter_calls(result):
    """Return Code Interpreter calls from one run."""
    return [
        item.raw_item
        for item in result.new_items
        if (
            isinstance(item, ToolCallItem)
            and getattr(item.raw_item, "type", None)
            == "code_interpreter_call"
        )
    ]


def used_code_interpreter(result) -> bool:
    """True when Code Interpreter completed at least one call."""
    return any(
        getattr(call, "status", None) == "completed"
        for call in get_code_interpreter_calls(result)
    )


def track_container_ids(result):
    """Record containers observed in one run for later cleanup."""
    for call in get_code_interpreter_calls(result):
        container_id = getattr(call, "container_id", None)
        if container_id:
            created_container_ids.append(container_id)


def container_file_citations(result):
    """Return generated-file annotations from assistant messages."""
    citations = []
    for item in result.new_items:
        if not isinstance(item, MessageOutputItem):
            continue
        for content in item.raw_item.content:
            for annotation in getattr(content, "annotations", []):
                if (
                    getattr(annotation, "type", None)
                    == "container_file_citation"
                ):
                    # Artifact retrieval uses IDs and filenames.
                    # Response-text spans are deliberately not used.
                    citations.append(annotation)
    return citations


def download_container_file(citation, destination):
    """Download one cited container file to a local path."""
    response = openai_client.containers.files.content.retrieve(
        file_id=citation.file_id,
        container_id=citation.container_id,
    )
    response.write_to_file(str(destination))


print("✅ Evidence helpers ready!")

---

## 🎯 The Problem

Language models can reason about numbers while still making arithmetic errors.

Code Interpreter lets an agent write and run Python instead of estimating.

The model still writes the code, so verify results that matter.

---

## 🖥️ Part 1: How Code Interpreter Works

When Code Interpreter is enabled:

1. The agent receives a task requiring computation

2. It writes Python code to solve it

3. OpenAI runs the code in an isolated sandbox container

4. The output returns to the agent for its final response

Treat containers as temporary.

They expire after 20 minutes of inactivity.

Download generated files while their container is active.

### 💡 Cost Note

Container sessions are billed separately from model tokens.

Check current rates in the
[official pricing guide](https://developers.openai.com/api/docs/pricing#built-in-tools).

---

## 🔢 Part 2: Math and Computation

The clearest use case is precise computation.

Both agents below receive the same instructions and question.

Only the second agent receives `CodeInterpreterTool`.

`CodeInterpreterTool` takes a `tool_config` dictionary.

Auto mode creates a container or reuses one from model context.

### Before: Without Code Interpreter

In [ ]:
comparison_instructions = (
    "You are a helpful assistant. Answer math and data questions accurately."
)
prime_question = (
    "How many prime numbers below 100,000 have digits that sum to exactly 17?"
)

no_code_agent = Agent(
    name="NoCodeAgent",
    instructions=comparison_instructions,
    model=MODEL,
)

result = await Runner.run(no_code_agent, input=prime_question)

unexpected_calls = get_code_interpreter_calls(result)
print("Without Code Interpreter:")
print(f"Code Interpreter calls: {len(unexpected_calls)}")
print(result.final_output)

### After: With Code Interpreter

In [ ]:
code_agent = Agent(
    name="CodeAgent",
    instructions=comparison_instructions,
    model=MODEL,
    tools=[CodeInterpreterTool(tool_config={
        "type": "code_interpreter",
        "container": {"type": "auto"},
    })],
)

result = await Runner.run(code_agent, input=prime_question)
track_container_ids(result)

completed = used_code_interpreter(result)
match_599 = bool(re.search(r"(?<![\d,])599(?!\d)", result.final_output))

print("With Code Interpreter:")
print(f"Code Interpreter completed: {completed}")
print(f"Matches the independently computed answer (599): {match_599}")
if not (completed and match_599):
    print("⚠️  Computation evidence is incomplete")
print(result.final_output)

### 💡 Key Takeaway

Compare important answers with values computed outside the agent.

---

## 📊 Part 3: Data Analysis and Generated Files

Small structured datasets can travel in the prompt as JSON.

The application can also retrieve files the sandbox generates.

In [ ]:
sales_records = [
    {"month": "January", "revenue": 42500, "units": 850},
    {"month": "February", "revenue": 38200, "units": 764},
    {"month": "March", "revenue": 51300, "units": 1026},
    {"month": "April", "revenue": 49800, "units": 996},
    {"month": "May", "revenue": 55100, "units": 1102},
    {"month": "June", "revenue": 61200, "units": 1224},
]
sales_json = json.dumps(sales_records, indent=2)

analysis_agent = Agent(
    name="DataAnalyst",
    instructions=(
        "You are a data analyst. Always use Code Interpreter to analyze "
        "data accurately and present clear summaries."
    ),
    model=MODEL,
    tools=[CodeInterpreterTool(tool_config={
        "type": "code_interpreter",
        "container": {"type": "auto"},
    })],
)

# --------------------------------------------------------------
print("✅ Data analyst agent ready")

#### Run Data Analysis

In [ ]:
analysis_request = f"""Analyze this JSON sales data with Python and report:
1. Total revenue and units for the period
2. Best and worst performing months by revenue
3. Average monthly revenue
4. Month-over-month revenue growth trend

JSON data:
{sales_json}"""

result = await Runner.run(analysis_agent, input=analysis_request)
track_container_ids(result)

out = result.final_output
completed = used_code_interpreter(result)
revenue_ok = bool(re.search(r"(?<![\d,])298,?100(?!\d)", out))
units_ok = bool(re.search(r"(?<![\d,])5,?962(?!\d)", out))

print("=" * 60)
print(f"Code Interpreter completed: {completed}")
print(
    "Matches revenue 298,100 and units 5,962: "
    f"{revenue_ok and units_ok}"
)
if not (completed and revenue_ok and units_ok):
    print("⚠️  Analysis evidence is incomplete")
print(out)
print("=" * 60)

#### Generate and Retrieve Files

The next run creates a PNG chart and a JSON summary in the sandbox.

The assistant returns generated files through structured citations.

The application downloads, parses, verifies, and displays those artifacts.

In [ ]:
plot_request = f"""Use Python to analyze the JSON sales data below.

Save a bar chart of monthly revenue to /mnt/data/monthly_revenue.png.
Label each bar with the month name.

Also save /mnt/data/monthly_revenue.json with exactly these keys:
total_revenue, total_units, best_month.

Link to both generated files in your response and briefly describe the chart.

JSON data:
{sales_json}"""

result = await Runner.run(analysis_agent, input=plot_request)
track_container_ids(result)

citations = container_file_citations(result)
png_citation = next(
    (item for item in citations if item.filename.endswith(".png")),
    None,
)
json_citation = next(
    (item for item in citations if item.filename.endswith(".json")),
    None,
)

demo_png_path = Path("monthly_revenue.png")
demo_json_path = Path("monthly_revenue.json")
completed = used_code_interpreter(result)
artifacts_ok = png_citation is not None and json_citation is not None
json_ok = False
png_ok = False

if artifacts_ok:
    download_container_file(png_citation, demo_png_path)
    download_container_file(json_citation, demo_json_path)

    summary = json.loads(demo_json_path.read_text())
    json_ok = (
        summary.get("total_revenue") == 298100
        and summary.get("total_units") == 5962
        and summary.get("best_month") == "June"
    )
    png_ok = demo_png_path.read_bytes().startswith(b"\x89PNG\r\n\x1a\n")

print("=" * 60)
print(f"Code Interpreter completed: {completed}")
print(f"Cited PNG and JSON: {artifacts_ok}")
print(f"Parsed JSON matches source data: {json_ok}")
print(f"Downloaded file is a PNG: {png_ok}")
if not (completed and artifacts_ok and json_ok and png_ok):
    print("⚠️  Generated-file evidence is incomplete")
else:
    display(Image(filename=str(demo_png_path)))
print(result.final_output)
print("=" * 60)

### 💡 Key Takeaway

Generated files matter only when your application retrieves and verifies them.

## 🧹 Demo Cleanup

The downloaded copies are no longer needed after the evidence is displayed.

In [ ]:
for demo_path in (demo_png_path, demo_json_path):
    if demo_path.exists():
        demo_path.unlink()
        print(f"✅ Local file removed: {demo_path.name}")

---

## 📁 Part 4: File Input

The container is separate from your machine.

Upload a local file through the Files API, then mount its file ID.

`FileSearchTool` and `CodeInterpreterTool` use different file backends.

This demo uploads to OpenAI's general file store with `purpose="assistants"`.

The `assistants` purpose comes from the Assistants API, but the uploaded
file's ID can be mounted in Code Interpreter.

⚠️ **Security note:** The sandbox isolates code from your machine.

It can still read mounted files and return their contents.

Upload only data the agent is allowed to process.

In [ ]:
csv_path = Path("sales_report.csv")

try:
    csv_path.write_text(
        "Month,Revenue,Units\n"
        "January,42500,850\n"
        "February,38200,764\n"
        "March,51300,1026\n"
        "April,49800,996\n"
        "May,55100,1102\n"
        "June,61200,1224\n"
    )

    with open(csv_path, "rb") as file_handle:
        uploaded_file = openai_client.files.create(
            file=file_handle,
            purpose="assistants",
        )
    created_file_ids.append(uploaded_file.id)
    print(f"✅ File uploaded: {uploaded_file.id}")
finally:
    if csv_path.exists():
        csv_path.unlink()
        print("✅ Local CSV removed")

#### Configure the File Analyst

Mount the uploaded file ID when the container is created.

In [ ]:
file_agent = Agent(
    name="FileAnalyst",
    instructions=(
        "You are a data analyst. Always use Code Interpreter to read the "
        "uploaded CSV and analyze it accurately."
    ),
    model=MODEL,
    tools=[CodeInterpreterTool(tool_config={
        "type": "code_interpreter",
        "container": {
            "type": "auto",
            "file_ids": [uploaded_file.id],
        },
    })],
)

# --------------------------------------------------------------
print("✅ File analyst agent ready")

#### Analyze the File

In [ ]:
file_question = (
    "Read sales_report.csv with Python. Report the total revenue, "
    "best month, and average monthly units."
)

result = await Runner.run(file_agent, input=file_question)
track_container_ids(result)

out = result.final_output
completed = used_code_interpreter(result)
total_ok = bool(re.search(r"(?<![\d,])298,?100(?!\d)", out))

print("=" * 60)
print(f"Code Interpreter completed: {completed}")
print(
    "Grounded in the CSV (total revenue 298,100): "
    f"{total_ok}"
)
if not (completed and total_ok):
    print("⚠️  Uploaded-file evidence is incomplete")
print(out)
print("=" * 60)

### 💡 Key Takeaway

Upload a file, then mount its file ID before the Code Interpreter run begins.

---

## 💪 Practice Exercises

### Exercise 1: Statistics Calculator

*Covers: `CodeInterpreterTool`, numerical analysis in Python*

Compute statistics for a fixed list and confirm known results.

Costs one paid agent run and may start a container session.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Statistics Calculator
# --------------------------------------------------------------
# Objective: Use Code Interpreter to compute accurate statistics.

numbers = [23, 45, 12, 67, 34, 89, 56, 78, 90, 11, 44, 66, 33, 55, 45]

# TODO 1: Create an Agent with CodeInterpreterTool
#         Instruct it to compute statistics precisely with Python

# TODO 2: Ask for mean, median, mode, standard deviation,
#         minimum, maximum, and range

# TODO 3: Print result.final_output and confirm:
#         - Code Interpreter completed
#         - median is 45
#         - mode is 45
#         - range is 79
#         Do not assert standard deviation unless you specify
#         whether it is the sample or population value

# --- Write your code below this line ---

### Exercise 2: Text Analyzer

*Covers: text analysis, generated JSON files, artifact retrieval*

Analyze text, export the top frequencies, and verify the cited JSON artifact.

Costs one paid agent run and may start a container session.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Text Analyzer
# --------------------------------------------------------------
# Objective: Analyze text and retrieve a generated JSON result.

sample_text = """
Artificial intelligence is transforming how we work and live. Machine learning,
a subset of artificial intelligence, enables computers to learn from data.
Deep learning, a subset of machine learning, uses neural networks to process
complex patterns. Artificial intelligence and machine learning are now used
in healthcare, finance, and transportation.
"""

# TODO 1: Create an Agent with CodeInterpreterTool

# TODO 2: Compute word frequencies and save the top five results
#         to word_frequencies.json in the sandbox

# TODO 3: Confirm a completed Code Interpreter call
#         Retrieve the cited JSON file and parse it with json.loads
#         Confirm the parsed result contains exactly five entries

# --- Write your code below this line ---

---

## 🧹 Cleanup: Delete Remote Resources

Run this cell after the exercises to remove tracked files and containers.

In [ ]:
if not created_file_ids:
    print("⚠️  No uploaded file IDs are recorded in this session.")
    print("   After a kernel restart, check the API dashboard for cleanup.")
else:
    remaining_file_ids = []
    for file_id in dict.fromkeys(created_file_ids):
        try:
            openai_client.files.delete(file_id)
            print(f"✅ Uploaded file deleted: {file_id}")
        except Exception as error:
            remaining_file_ids.append(file_id)
            print(f"⚠️  File cleanup failed: {type(error).__name__}")
    created_file_ids[:] = remaining_file_ids

if not created_container_ids:
    print("⚠️  No container IDs are recorded in this session.")
    print("   After a kernel restart, check the API dashboard for cleanup.")
else:
    remaining_container_ids = []
    for container_id in dict.fromkeys(created_container_ids):
        try:
            openai_client.containers.delete(container_id)
            print(f"✅ Container deleted: {container_id}")
        except Exception as error:
            remaining_container_ids.append(container_id)
            print(f"⚠️  Container cleanup failed: {type(error).__name__}")
    created_container_ids[:] = remaining_container_ids

---

## 🎯 Key Takeaways

**Execution needs evidence:**

- Require a completed `code_interpreter_call`, not merely a call item

- Check important calculations independently

- Treat the final explanation as model-authored text
<br>
<br>

**Data crosses the sandbox boundary explicitly:**

- Small JSON can travel in the prompt

- Uploaded CSV files enter through file IDs

- Generated files return through container-file citations
<br>
<br>

**Containers are temporary resources:**

- Independent auto-mode runs may create separate container sessions

- Download generated files before their container expires

- Delete tracked files and containers after the exercises

---

## 📍 Next Step

**Notebook 11: Capstone #1**  

Combine File Search and Code Interpreter in one research agent.

It retrieves two documents, runs the numbers, and returns a structured
report.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-10-code-interpreter)

---